# Qwen3.8-27B → DeepSeek Harness

This notebook uses the outbound-only Supabase relay. No reverse tunnel or public Colab port is used.

In Colab **Secrets** (key icon), add `QWEN_RELAY_SECRET` and enable **Notebook access**.

For automatic Oracle VM wake-up, also add `ORACLE_WAKE_GITHUB_TOKEN` and enable **Notebook access**. Use a fine-grained token restricted to the relay repository with only the required Actions permission.

### Cheapest validation strategy
Before paying for A100 time, use **Section 2A — Expected-OOM smoke test** on the cheapest compatible **SM80+** GPU you can get. **L4 24 GB is recommended.** It deliberately runs the real 27B FP8 + 262K + MTP3 production stack until VRAM/KV allocation fails. Reaching the expected memory limit is a **PASS**. A T4 is not suitable for this proof because its older compute capability can fail on FP8/Marlin compatibility before the intended OOM.

Use **Section 2B** only for the final A100 fit/performance test. Do not run Sections 2A, 2B and 3 at the same time.


## Section 1 — Repair Colab Python packages + install/update worker
This first repairs Pillow as a clean package because some Colab images can contain mixed `PIL` files that fail with `cannot import name _Ink from PIL._typing`. Then it installs the latest worker from GitHub. If this runtime has **already** shown the `_Ink` ImportError, run Section 1, choose **Runtime → Restart session**, then continue with Section 2A or 2B.


In [ ]:
%pip uninstall -y Pillow >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall Pillow
%pip install -q --upgrade --force-reinstall "git+https://github.com/Logan17de/All-testing.git#subdirectory=llm"
from PIL import ImageText
import PIL
print(f"Pillow {PIL.__version__}: OK ✅")


## Section 2A — CHEAP expected-OOM production-path smoke test
Use this **before A100**. Recommended runtime: **L4 24 GB** or another undersized SM80+ GPU. This runs the **same official Qwen3.8-27B-FP8 model, 262,144 context, FP8 KV, MTP3, FlashInfer, Marlin, CUDA graphs/torch.compile, Supabase preflight and Oracle lease** as production. The only relaxed rule is the 80 GB VRAM guard.

Expected result: vLLM reaches real model/KV initialization and fails because the GPU is too small. The cell converts that expected memory failure into `EXPECTED VRAM LIMIT REACHED ✅ — SMOKE TEST PASSED`. If you get any other error first, that is a real code/dependency/configuration problem to fix before spending A100 time.

**Do not use a T4 for this proof.** T4 can fail on production-kernel compatibility before OOM.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_oom_smoke as qwen_oom_smoke
importlib.reload(qwen_oom_smoke)
qwen_oom_smoke.main()


## Optional — capture BEFORE benchmark
Use this only when doing the final A100 comparison and the old BF16 vLLM server is still running in the same runtime. It measures localhost TTFT and decode tokens/sec and saves the result under `/content/qwen_benchmark_results.json`.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_fast_nightly as qwen_fast
importlib.reload(qwen_fast)
qwen_fast.benchmark_running_server("before")


## Section 2B — REAL Qwen3.8-27B optimized worker (A100 80 GB)
Use this only after Section 2A reaches the expected VRAM limit cleanly. This uses official `Qwen/Qwen3.8-27B-FP8` weights, native 262,144 context, FP8 KV cache, native MTP with 3 draft tokens, prefix caching, chunked prefill, torch.compile/CUDA Graphs, and single-user scheduling. It installs the CUDA-13 vLLM nightly with the Qwen3.8 gated-DeltaNet MTP fix using `uv`. On A100, the profile uses FlashInfer for FP8-KV-compatible attention and Marlin for FP8 W8A16 linear layers. The local benchmark runs before the worker is advertised ready and reports TTFT, output tok/s and MTP acceptance.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_fast_nightly as qwen_worker
importlib.reload(qwen_worker)
qwen_worker.main()


## Section 3 — TESTING: API/relay only (NO GPU)
Use a normal CPU Colab runtime. Run **Section 1**, skip Sections 2A and 2B, then run this section. No Torch, vLLM, CUDA, Hugging Face model, or GPU is used. Every request received from Harness is decoded through the real relay and returned as the OpenAI-compatible assistant response **`succeed`**. Leave this cell running while testing Harness.


In [ ]:
import importlib
import qwen_supabase_test_worker as relay_test
importlib.reload(relay_test)
relay_test.main()
